# Step 7B — Prepare a manual sentiment review

We select 30 articles from the previously fixed validation sample and compare their supplied sentiment scores with human judgments. This review does not alter the raw dataset.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "apple_news_data.csv"
SAMPLE_PATH = PROJECT_ROOT / "data" / "processed" / "apple_headline_validation_sample.csv"
REVIEW_PATH = PROJECT_ROOT / "data" / "processed" / "apple_sentiment_manual_review.csv"

raw = pd.read_csv(RAW_PATH)
sample = pd.read_csv(SAMPLE_PATH)

## Select articles reproducibly

We randomly choose 30 rows with a fixed seed and recover their full article content by matching the unique source link.

In [2]:
selected = sample.sample(n=30, random_state=17).copy()
selected = selected.merge(raw[["link", "content"]], on="link", how="left", validate="one_to_one")
selected = selected.sort_values("published_at").reset_index(drop=True)

selected["dataset_label"] = "Neutral"
selected.loc[selected["sentiment_polarity"] > 0, "dataset_label"] = "Positive"
selected.loc[selected["sentiment_polarity"] < 0, "dataset_label"] = "Negative"
selected["content_excerpt"] = (
    selected["content"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip().str.slice(0, 500)
)
selected["manual_relevance"] = ""
selected["manual_sentiment"] = ""
selected["review_notes"] = ""
selected.insert(0, "review_id", range(1, len(selected) + 1))

## Review instructions

For each row:

- Set `manual_relevance` to **Relevant**, **Partly relevant**, or **Irrelevant** based on whether the article discusses information that could affect perceptions of Apple.
- Set `manual_sentiment` to **Positive**, **Neutral**, or **Negative**, judging the article's tone toward Apple—not whether every word sounds positive or negative.
- Use `review_notes` for ambiguity, mixed tone, misleading headlines, or disagreement with the dataset label.

In [3]:
review_columns = [
    "review_id", "published_at", "title", "content_excerpt",
    "sentiment_polarity", "sentiment_neg", "sentiment_neu", "sentiment_pos",
    "dataset_label", "manual_relevance", "manual_sentiment", "review_notes", "link",
]
review = selected[review_columns]
review.to_csv(REVIEW_PATH, index=False)
print(f"Saved {len(review)} review rows to {REVIEW_PATH}")
review["dataset_label"].value_counts()

Saved 30 review rows to /Users/keishakalra/Desktop/Financial_App/data/processed/apple_sentiment_manual_review.csv


dataset_label
Positive    22
Negative     4
Neutral      4
Name: count, dtype: int64

In [4]:
pd.set_option("display.max_colwidth", 120)
review[["review_id", "title", "content_excerpt", "dataset_label", "sentiment_polarity"]].head(10)

,review_id,title,content_excerpt,dataset_label,sentiment_polarity
0,1,More than $1 trillion wiped off value of Apple in face of China chaos,apple store More than $1 trillion has been wiped off the value of Apple since its peak as tech stocks are roiled by ...,Negative,-0.883
1,2,"Stocks open lower, Tesla stock climbs as Apple and Microsoft slide",Yahoo Finance Live’s Brad Smith discusses how markets opened on Monday.,Neutral,0.000
2,3,This Is Retail Investors' Favorite Stock to Own (and It's Not Apple or a Meme Stock),Everyday investors have stomped on the accelerator to get this high-growth stock into their portfolios. Continue rea...,Neutral,0.000
3,4,Beyond the iPhone: Here's What May Decide Apple's Future,The company will need to rely on its service offerings to continue growing at the current pace over the long term. C...,Positive,0.178
4,5,Shares of Apple suppliers fall on reports of China iPhone curbs,"TAIPEI (Reuters) -The shares of several major Apple suppliers fell on Friday, following reports that China had widen...",Positive,0.778
5,6,"iPhone 15 launch: Release date, price and new features",Apple will host its iPhone 15 launch event called “Wanderlust” in California - APPLE INC HANDOUT/EPA-EFE/Shutterstoc...,Positive,0.897
6,7,Apple's Superpower? It's About the Ecosystem.,"Apple's success isn't just about one product, it's about how many products work together.",Positive,0.572
7,8,Apple Inc. (NASDAQ:AAPL) is largely controlled by institutional shareholders who own 55% of the company,Key Insights Institutions' substantial holdings in Apple implies that they have significant influence over the compa...,Positive,0.994
8,9,The company that makes your iPhone is expanding to EVs and it’s getting Nvidia to help make an ‘AI factory’,"Foxconn's annual showcase on Wednesday featured a special guest: Nvidia CEO Jensen Huang, clad in his now-famous lea...",Positive,0.991
9,10,Apple Unveils M3 Processors and New MacBook Pros,A mysterious Apple launch event is offering hope of some positive news ahead of earnings but Wall Street analysts ar...,Positive,0.392
